In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualizacao grafica inline no Jupyter notebook
%matplotlib inline

# Treinar uma linha de base segura contra vazamento (*leakage-safe*)

**Dificuldade 1-2** | **Tempo de execucao: 30s** | **Computacao: CPU**

Um modelo que alcanca 0.78 de acuracia em janelas retidas (*held-out*) so e util quando voce tambem sabe como sao os desempenhos de 0.50 (acaso) e 0.55 (uma linha de base linear transparente) na mesma divisao de dados. Este tutorial treina essa linha de base linear em tres participantes do OpenNeuro ``ds002718`` (Wakeman & Henson 2015), acessivel atraves do [NEMAR](https://nemar.org) (Delorme et al. 2022). Quatro bandas de potencia logaritmica por canal alimentam um :class:`sklearn.linear_model.LogisticRegression` (Pedregosa et al. 2011); um loop de validacao cruzada inter-sujeito em 3 particoes com :class:`sklearn.model_selection.GroupKFold` mantem cada participante em exatamente uma particao de teste. O resultado e uma figura de tres paineis que responde a tres perguntas em uma unica tela: as caracteristicas separam as classes? Como a acuracia varia entre os sujeitos retidos? Quais ensaios o modelo confunde?

.. sphinx_gallery_thumbnail_path = '_static/thumbs/plot_12_train_a_baseline.png'
Palavras-chave: classificacao, linha de base, avaliacao


## Objetivos de aprendizagem
- Calcular caracteristicas de potencia logaritmica em quatro bandas canonicas (teta, alfa, beta, gama) por canal a partir de janelas alinhadas a eventos de um conjunto de dados BIDS real.
- Executar um loop de validacao cruzada inter-sujeito em 3 particoes com :class:`~sklearn.model_selection.GroupKFold` para que um participante nunca apareca simultaneamente no treino e no teste.
- Ajustar uma linha de base com :class:`~sklearn.linear_model.LogisticRegression` e obter a acuracia por particao, media +/- desvio padrao e uma matriz de confusao normalizada por linha :func:`~sklearn.metrics.confusion_matrix` a partir da mesma execucao.
- Comparar esses numeros contra a linha de base de classe majoritaria (*majority_baseline*) na mesma divisao.
- Produzir uma figura diagnostica de tres paineis que permita julgar a linha de base de relance.



## Requisitos
- Cerca de 90 s em CPU na primeira execucao; menos de 30 s uma vez em cache.
- Rede na primeira chamada (~30 MB baixados em ``cache_dir``); offline posteriormente.
- Pre-requisitos: :doc:`plot_11_leakage_safe_split` (divisoes inter-sujeito), :doc:`plot_10_preprocess_and_window` (janelamento por eventos).
- Conceito: :doc:`/concepts/features_vs_deep_learning`.



## Por que uma linha de base antes de uma rede neural profunda?
Uma linha de base e um numero defensavel em uma revisao de codigo. A regressao logistica sobre potencia em bandas possui tres propriedades que redes caixa-preta nao oferecem: cada coeficiente mapeia diretamente para um par (canal, banda), todo o pipeline cabe em poucas linhas e o tempo de execucao permanece dentro de um orcamento compativel com CPU. Cisotto & Chicco 2024 enquadram isso na Dica 5: um classificador compreensivel com acuracia de 0.62 e mais util para a literatura do que um modelo opaco com 0.71. A linha de base linear tambem funciona como delimitador critico: uma rede profunda que falha em superar a barra linear geralmente sofre com vazamento de dados ou falhas de rotulagem, e nao com falta de capacidade de representacao :cite:`schirrmeister2017braindecode`.

Tres razoes para estruturarmos o loop inter-sujeito *antes* de qualquer engenharia de caracteristicas:

- **O sujeito como fator de confusao:** A amplitude do EEG difere mais entre participantes do que entre condicoes cognitivas. Uma divisao intra-sujeito superestima essa variancia e infla a acuracia.
- **A unidade de generalizacao e o participante:** A pergunta central de benchmark e "o modelo se generaliza para uma nova pessoa?", e nao "ele memoriza essa pessoa?".
- **A divisao fixa o nivel de chance:** A acuracia da classe majoritaria e calculada estritamente no conjunto de teste retido, garantindo que o nivel de chance informado reflita o equilibrio real da particao de teste.

## Valide seu resultado
- **Acuracia:** Espere que a linha de base linear pontue consideravelmente acima do acaso (por exemplo, 0.60-0.75 para P300 visual), mas abaixo de um modelo profundo bem ajustado.
- **Nivel de chance:** Verifique se o valor de referencia coincide com o desbalanceamento do conjunto de dados.
- **Matriz de confusao:** A matriz normalizada por linha deve exibir dominancia na diagonal principal caso o modelo tenha aprendido a tarefa.



Configuracao inicial. Definir ``random_state=42`` em todos os estimadores e divisores mantem as acuracias reprodutiveis (E3.21).



In [ ]:
# Importa utilitarios de sistema operacional e avisos
import os
import warnings
from pathlib import Path

# Importa bibliotecas para graficos, MNE para eletrofisiologia, arrays numericos e DataFrames
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
# Importa rotinas de pre-processamento e janelamento da Braindecode
from braindecode.preprocessing import (
    Preprocessor,
    create_windows_from_events,
    preprocess,
)
# Importa regressor logistico, metricas, GroupKFold, pipeline e padronizador do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Importa utilitarios de contagem e classes do EEGDash
from collections import Counter

import eegdash
from eegdash import EEGDashDataset
from eegdash.viz import use_eegdash_style

# Configura estilo visual, nivel de log e semente aleatoria
use_eegdash_style()
mne.set_log_level("ERROR")
warnings.simplefilter("ignore", category=RuntimeWarning)
SEED = 42
CACHE_DIR = Path(os.environ.get("EEGDASH_CACHE_DIR", Path.home() / ".eegdash_cache"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"eegdash {eegdash.__version__} | cache_dir={CACHE_DIR}")

## Etapa 1: Carregar tres participantes de ds002718
**Preveja.** ``ds002718`` e um estudo de percepcao de faces com tres condicoes: ``famous`` (rostos famosos), ``unfamiliar`` (rostos desconhecidos) e ``scrambled`` (rostos embaralhados) (Wakeman & Henson 2015). Contrastamos ``famous`` contra ``scrambled`` para manter as classes equilibradas e a chance em 0.50; eventos de faces desconhecidas sao descartados para evitar um desbalanceamento de 2:1 que inflaria o acaso para 0.67. Tres sujeitos (``002``, ``003``, ``004``) mantem a execucao rapida enquanto fornecem participantes suficientes para 3 particoes de validacao cruzada inter-sujeito.



In [ ]:
# Define conjunto de dados, sujeitos e tarefa experimental
DATASET = "ds002718"
SUBJECTS = ["002", "003", "004"]
TASK = "FaceRecognition"
# Instancia o dataset carregando as gravacoes dos 3 participantes
dataset = EEGDashDataset(
    cache_dir=CACHE_DIR, dataset=DATASET, subject=SUBJECTS, task=TASK
)
# Monta resumo descritivo das gravacoes carregadas
records_summary = pd.Series(
    {
        "n_recordings": len(dataset.datasets),
        "subjects": ", ".join(SUBJECTS),
        "raw n_channels": dataset.datasets[0].raw.info["nchan"],
        "raw sfreq (Hz)": float(dataset.datasets[0].raw.info["sfreq"]),
    },
    name="value",
).to_frame()
records_summary

## Descoberta de anotacoes: quais eventos estao realmente no arquivo?
**Execute.** Antes de mapear eventos para inteiros, conte os marcadores em :attr:`mne.io.Raw.annotations`. Fixar mapeamentos presumidos sem inspecionar os rotulos originais e a causa mais comum de conjuntos de janelas vazios.



In [ ]:
# Lista para armazenar descricoes literais de todos os eventos nas gravacoes
descriptions: list[str] = []
for record in dataset.datasets:
    descriptions.extend(record.raw.annotations.description.tolist())
# Conta a frequencia de cada tipo de evento presente
event_counts = (
    pd.Series(descriptions, name="description")
    .value_counts()
    .rename_axis("description")
    .to_frame(name="count")
)
# Exibe os 12 tipos de eventos mais frequentes
event_counts.head(12)

**Investigue.** A coluna de tipos de ensaio contem rotulos detalhados: ``famous_new``, ``famous_second_early``, ``famous_second_late``, ``unfamiliar_new``, ``scrambled_new``, ``scrambled_second_late``, alem de marcadores de pressao de botao. Para fixar a referencia do acaso em 0.50, agrupamos as tres condicoes de faces famosas na classe ``0`` e as tres condicoes de faces embaralhadas na classe ``1``, descartando faces desconhecidas e cliques de resposta.



## Etapa 2: Dois pre-processadores seguros e janelamento por eventos
Dois pre-processadores mantem o processamento previsivel: selecionar canais de EEG e reamostrar para 100 Hz. O janelamento alinhado a eventos e executado com :func:`braindecode.preprocessing.create_windows_from_events`. Cada janela cobre 1 segundo apos o inicio do estimulo (``trial_start_offset_samples = 0``, ``trial_stop_offset_samples = sfreq``).



In [ ]:
# Define frequencia de amostragem alvo e duracao da janela
TARGET_SFREQ = 100  # Hz
WINDOW_SECONDS = 1.0
# Aplica pre-processamento selecionando canais EEG e reamostrando o sinal
preprocess(
    dataset,
    [
        Preprocessor("pick_types", eeg=True, eog=False, misc=False),
        Preprocessor("resample", sfreq=TARGET_SFREQ),
    ],
)
window_size_samples = int(WINDOW_SECONDS * TARGET_SFREQ)
# Mapeamento explicito das anotacoes originais para as classes binarias 0 e 1
EVENT_MAPPING = {
    "famous_new": 0,
    "famous_second_early": 0,
    "famous_second_late": 0,
    "scrambled_new": 1,
    "scrambled_second_early": 1,
    "scrambled_second_late": 1,
}
CLASS_NAMES = ("famous", "scrambled")
# Cria janelas temporais de 1 segundo alinhadas ao inicio de cada estimulo valido
windows = create_windows_from_events(
    dataset,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=window_size_samples,
    preload=True,
    drop_bad_windows=True,
    mapping=EVENT_MAPPING,
)
# Extrai metadados dos canais e taxa das janelas geradas
ch_names = list(windows.datasets[0].windows.info["ch_names"])
n_channels = len(ch_names)
sfreq = float(windows.datasets[0].windows.info["sfreq"])
print(
    f"n_windows={len(windows)} | n_channels={n_channels} | "
    f"sfreq={sfreq:.0f} Hz | window_size_samples={window_size_samples}"
)

## Etapa 3: Materializar janelas e identificadores de sujeitos por janela
O divisor inter-sujeito requer um array ``groups`` com o identificador de participante de cada janela para isolar os dados nos folds.



In [ ]:
# Listas para consolidar tensores de sinal, classes alvo e grupos de sujeitos
X_list: list[np.ndarray] = []
y_list: list[int] = []
groups_list: list[str] = []
# Itera por cada subdataset extraindo janelas e metadados
for sub_ds in windows.datasets:
    subj = str(sub_ds.description.get("subject"))
    for k in range(len(sub_ds)):
        x_k, y_k, _ = sub_ds[k]
        X_list.append(np.asarray(x_k, dtype=np.float32))
        y_list.append(int(y_k))
        groups_list.append(subj)
# Converte para arrays numpy
X = np.stack(X_list)
y = np.asarray(y_list, dtype=int)
groups = np.asarray(groups_list)

**Preveja.** ``X.shape`` deve ser ``(n_windows, n_channels, window_size_samples)``. As contagens em ``np.bincount(y)`` devem ficar proximas de 1:1.



In [ ]:
# Exibe resumo das dimensoes do tensor de entrada e distribuicao das classes
shape_summary = pd.Series(
    {
        "X.shape": str(X.shape),
        "X.dtype": str(X.dtype),
        "n_famous windows": int((y == 0).sum()),
        "n_scrambled windows": int((y == 1).sum()),
        "subjects in groups": ", ".join(sorted(set(groups))),
    },
    name="value",
).to_frame()
shape_summary

## Etapa 4: Calcular caracteristicas de potencia logaritmica em bandas
Para cada janela, o vetor de caracteristicas e composto por um valor de potencia em log por canal de EEG e por banda: teta (4-8 Hz), alfa (8-13 Hz), beta (13-30 Hz) e gama (30-45 Hz). O formato da matriz e ``(n_windows, n_bands * n_channels)``.



In [ ]:
# Definicao das quatro bandas espectrais de interesse
BANDS: tuple[tuple[float, float], ...] = (
    (4.0, 8.0),  # teta
    (8.0, 13.0),  # alfa
    (13.0, 30.0),  # beta
    (30.0, 45.0),  # gama
)
BAND_NAMES = ("theta", "alpha", "beta", "gamma")


# Funcao para extrair potencia espectral em escala logaritmica via FFT
def log_band_power(
    X_t: np.ndarray, sfreq: float, bands: tuple[tuple[float, float], ...]
) -> np.ndarray:
    """Calcula caracteristicas de potencia espectral logaritmica por canal em bandas."""
    # FFT real ao longo do eixo temporal
    spec = np.fft.rfft(X_t, axis=-1)
    power = (np.abs(spec) ** 2) / X_t.shape[-1]
    freqs = np.fft.rfftfreq(X_t.shape[-1], d=1.0 / sfreq)
    feats = []
    for fmin, fmax in bands:
        band_mask = (freqs >= fmin) & (freqs < fmax)
        # Piso numerico de 1e-12 para evitar log(0)
        feats.append(np.log(power[..., band_mask].mean(axis=-1) + 1e-12))
    return np.concatenate(feats, axis=-1).astype(np.float32)


# Calcula matriz de caracteristicas F
F = log_band_power(X, sfreq, BANDS)
print(
    f"feature matrix={F.shape} (log-power per channel for "
    f"{', '.join(BAND_NAMES)}) | dtype={F.dtype}"
)

## Inspecao descritiva das caracteristicas
Uma tabela descritiva rapida permite identificar canais sem variacao ou valores anomalos antes da classificacao.



In [ ]:
# Nomeia colunas associando banda e eletrodo
feature_names = [f"{band}_{ch}" for band in BAND_NAMES for ch in ch_names]
features_df = pd.DataFrame(F, columns=feature_names)
# Exibe estatisticas descritivas das primeiras 8 caracteristicas
features_df.iloc[:, :8].describe().round(3)

## Etapa 5: Validacao cruzada inter-sujeito em 3 particoes com GroupKFold
**Preveja.** Com tres participantes e ``GroupKFold(n_splits=3)``, cada particao treina em dois sujeitos e testa no terceiro mantido fora (*held-out*).



In [ ]:
# Configura o divisor GroupKFold com 3 particoes
N_FOLDS = 3
splitter = GroupKFold(n_splits=N_FOLDS)

# Estruturas para armazenar acuracias e atribuicoes por particao
fold_accuracies: list[float] = []
fold_held_out: list[str] = []
fold_chance: list[float] = []
fold_assignment = np.full(len(y), -1, dtype=int)
pooled_y_true: list[np.ndarray] = []
pooled_y_pred: list[np.ndarray] = []

# Executa o loop de validacao cruzada inter-sujeito
for fold_idx, (train_idx, test_idx) in enumerate(splitter.split(F, y, groups=groups)):
    held_out = sorted(set(groups[test_idx].tolist()))
    fold_held_out.append(held_out[0])
    fold_assignment[test_idx] = fold_idx

    # Cria pipeline com padronizador ajustado apenas no treino e regressao logistica
    clf = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(random_state=SEED, max_iter=2000)),
        ]
    )
    clf.fit(F[train_idx], y[train_idx])
    y_pred = clf.predict(F[test_idx])
    acc = float(accuracy_score(y[test_idx], y_pred))
    fold_accuracies.append(acc)
    # Calcula nivel de chance pela proporcao da classe majoritaria na particao de teste
    fold_chance.append(
        float(max(Counter(y[test_idx].tolist()).values()) / max(len(y[test_idx]), 1))
    )
    pooled_y_true.append(np.asarray(y[test_idx]))
    pooled_y_pred.append(np.asarray(y_pred))

# Concatena predicoes de todas as particoes de teste
y_true_pooled = np.concatenate(pooled_y_true)
y_pred_pooled = np.concatenate(pooled_y_pred)

# Calcula metricas globais medias
mean_acc = float(np.mean(fold_accuracies))
std_acc = float(np.std(fold_accuracies, ddof=0))
chance_overall = float(np.mean(fold_chance))
print(
    f"cross-subject CV: mean={mean_acc:.3f} +/- {std_acc:.3f} | "
    f"chance={chance_overall:.3f} | folds={N_FOLDS}"
)

## Tabela de resultados: acuracia por particao vs. acaso
Uma linha por particao para explicitar o sujeito retido e a diferenca (*lift*) em relacao ao acaso.



In [ ]:
# Monta tabela com acuracia individual de cada sujeito retido
results_df = pd.DataFrame(
    {
        "fold": np.arange(1, N_FOLDS + 1),
        "held-out subject": [f"sub-{sid}" for sid in fold_held_out],
        "accuracy": np.round(fold_accuracies, 3),
        "chance": np.round(fold_chance, 3),
        "lift": np.round(np.asarray(fold_accuracies) - np.asarray(fold_chance), 3),
    }
).set_index("fold")
results_df

## Erro comum: treinar na particao de teste por engano
Ajustar o pipeline nos dados de teste produz acuracias artificialmente infladas. Disparamos o erro deliberadamente para evidenciar a discrepancia.



In [ ]:
try:
    sneaky_train_idx, sneaky_test_idx = next(splitter.split(F, y, groups=groups))
    # Erro proposital: ajuste e avaliacao no conjunto de teste
    sneaky_clf = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(random_state=SEED, max_iter=2000)),
        ]
    )
    sneaky_clf.fit(F[sneaky_test_idx], y[sneaky_test_idx])
    sneaky_acc = float(
        accuracy_score(y[sneaky_test_idx], sneaky_clf.predict(F[sneaky_test_idx]))
    )
    # Correcao: ajuste no treino e avaliacao no teste
    recovered_clf = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(random_state=SEED, max_iter=2000)),
        ]
    )
    recovered_clf.fit(F[sneaky_train_idx], y[sneaky_train_idx])
    recovered_acc = float(
        accuracy_score(y[sneaky_test_idx], recovered_clf.predict(F[sneaky_test_idx]))
    )
    print(
        f"Train-on-test (wrong) acc={sneaky_acc:.2f} | "
        f"train-on-train (correct) acc={recovered_acc:.2f} | "
        f"gap={sneaky_acc - recovered_acc:+.2f}"
    )
except ValueError as exc:
    print(f"Caught ValueError: {str(exc)[:100]}")

## Painel diagnostico de tres partes
O grafico combina dispersao PCA das caracteristicas, desempenho das barras por particao e matriz de confusao agregada normalizada.



In [ ]:
# Importa rotina de plotagem do painel diagnostico
from _baseline_diagnostic import draw_baseline_diagnostic

# Desenha e exibe o painel diagnostico da linha de base
fig = draw_baseline_diagnostic(
    X_features=F,
    y_classes=y,
    fold_assignment=fold_assignment,
    fold_accuracies=fold_accuracies,
    y_true_pooled=y_true_pooled,
    y_pred_pooled=y_pred_pooled,
    class_names=CLASS_NAMES,
    subjects=SUBJECTS,
    held_out_subjects=fold_held_out,
    chance_level=chance_overall,
    plot_id="plot_12",
)
plt.show()

## Conclusao e referencias
Treinamos uma linha de base linear em 3 sujeitos sem vazamento de dados, mensurando o ganho em relacao ao acaso com metricas auditadas.

